# 🌾 Kaggriculture: The Farm You Inherited
### *A beginner's story in three seasons — from one melon seed to a grandmaster empire*

---

**The letter arrives on a Monday morning.**

> *"Dear cousin — the farm is yours now. $3,000 in the coffee tin, a 5×5 plot of dirt, and one growing season: **30 days**. The town will buy anything you grow. Across the road, your rival inherited the other farm. Rule of the will: **whoever has more coins in the bank when day 30 ends keeps everything.** — Uncle Robert"*

**That's the whole game.** Kaggriculture is a 1-vs-1 farming duel: 30 days × 24 turns = **720 turns**. Each turn, you and every worker you've hired take **one action** (move, plant, water, fertilize, harvest, feed, build), and you may queue up to **10 market orders** (buy seeds, sell produce, hire help, buy land). At dawn on day 30, **whoever holds the most coins wins**. Crops still sitting in the barn? Worth *nothing*.

> **How this notebook tells the story.** You'll play **three seasons** — each one a smarter farmer than the last. Every agent is a full, working farmer you can submit to Kaggle, and each one fixes the mistakes of the one before:
>
> | Season | The farmer you become | What you learn | Typical bank vs `random` |
> |---|---|---|---|
> | 1 | 🍈 **The Melon Dreamer** | the basics: one crop, hard work, sell high | ~$6,000 |
> | 2 | 🥕 **The Market Gardener** | read prices, grow a mix, sell in small baskets | ~$25,000 |
> | 3 | 🏭 **The Farm Empress** | land, workers, geese, and a market math engine | **~$43,000** |
>
> Scores wiggle from game to game (weather of the market, opponents, seeds) — treat them as ballparks, not promises. Across **two independent batches of local tournaments on unseen seeds** the Farm Empress beat **every** opponent class — `random` 12–0, `starter` 12–0, the Season-1 farmer 7–0, the Season-2 farmer 7–0 — and beat the previous champion Empress (v4) in **11 of 19** head-to-head games while losing none of the robustness guarantees.

**Your three seasons**

```
Season 1 ── learn the rhythm: observe → decide → act
   │
Season 2 ── + score crops by profit      + sell in baskets (tranches)
   │        + price floors               + harvest at the perfect day
Season 3 ── + buy land & hire workers    + the Goose Empire (eggs + free fertilizer)
   │        + fertilizer timing windows  + market-depth math (the melon race)
            + scout the rival's farm     + the day-30 sweep
```

Run the notebook top-to-bottom and watch each season play out. The last chapter writes **`submission.py`** — your ticket to the Kaggle ladder, where the top 10 farmers split **$50,000**.

## 📖 Chapter 0 — The Farmer's Almanac
### *Four laws Uncle Robert scribbled on the back of the letter. Read them once; they're worth thousands.*

> **Law 1 — Your real currency is *time*, not coins.**
> Every worker takes **1 action per turn, 24 turns a day**. One crop tile needs about one action a day (watering) plus planting and harvesting. So one farmer can truly care for **~15–20 tiles** — your 5×5 home plot is already at the edge of one person's stamina. Every leap in income comes from two purchases: **hands** (villagers who work for absurdly cheap Fibonacci wages: $1, $1, $2, $3, $5, $8… but they *vanish every night* and must be re-hired each morning) and **land** ($1K / $2K / $4K per 25-tile quadrant).

> **Law 2 — Crops are employees; judge them by profit per tile-day.**
>
> | Crop | Seed | Base price | Ready in | Typical yield* | **$/tile-day** |
> |---|---|---|---|---|---|
> | 🌾 Wheat | $10 | $25 | 4 d | 6 (fertilized) | *(6·25−10)/5 = **$28*** |
> | 🥕 Carrot | $20 | $35 | 3 d | 4 (fertilized) | *(4·35−20)/4 = **$30*** |
> | 🍅 Tomato | $50 | $60 | 8 d | 8 (fertilized) | **~$50/cycle** |
> | 🍓 Strawberry | $100 | $120 | 10 d | 8 (fertilized) | **~$86/cycle** |
> | 🍈 Melon | $80 | $250 | 10 d | 6 | *(6·250−80)/11 = **$129*** |
>
> \* "Fertilized" numbers use the secret fertilizer trick — see Law 2½ below. Formula: `(yield × price − seed) / days`.

> **Law 2½ — Fertilizer is an *energy drink for watering days*, not plant food.**
> Watering a crop **inside its yield window** adds +1 yield (+2 if the plant is fertilized that day). Every window is short: melon drinks on **days 6–11**, carrots on **days 2–3**, wheat on **days 2–4**; tomatoes pay out on **days 8–11**, strawberries on **days 10–16**. Fertilize on the *first day of the window* and one $100 jug doubles a whole harvest. Animals make 1 free jug per day — remember that for Season 3.

> **Law 3 — The market is a stomach. Fill it and prices choke; let it digest and they breathe again.**
> Every product starts with 10,000 units on the shelf. Sell above that and the price slides down a curve — each curve has its own temper:
>
> | Product | Temperament |
> |---|---|
> | **Melon** | 🚨 The town center eats ~1 melon a day and *no shop serves melon pie*. Fill the stomach with ~160 melons and the price hits $1 — **and it never comes back**. A glut is forever. |
> | **Wheat, eggs** | The town devours them daily. Gluts *recover*. Sell in steady streams. |
> | **Milk, strawberries, carrots** | Shops eat them too, but the first 50–100 units of a glut still hurt. Small baskets. |
> | **Wool** | A mini-melon: quadratic crash. Sell early, never bank it for late game. |
>
> So: **goods the town digests → don't fear a glut, it breathes again. Melon → a glut is forever.** But *how fast* should you sell the digestible goods? Season 2 answers with baskets and floor prices… and Season 3 re-runs the math, discovers a surprising truth, and fires its own rule. Watch for the twist.

> **Law 4 — The buzzer erases everything you didn't sell.**
> Goods rot in pockets just as they rot on shelves: a harvested melon only becomes sellable after a worker **DROP**s it at the barn (100-shelf limit — overflow at dusk is *thrown away*). From day 28: stop buying, sell at any price, march every carrier to the barn. Also mind the planting clock — a melon seeded after day 15 can't safely ripen before the buzzer.

**Words the farm speaks (glossary)**

| Term | Meaning |
|---|---|
| **Turn / day** | 720 turns; `day = step // 24`, `hour = step % 24` |
| **Shed (barn)** | shared stockpile by the spawn; caps at **100 items** (seeds excluded); pockets auto-pour into it at dusk |
| **Yield window** | the days when watering actually adds yield (Law 2½) |
| **Ongoing crop** | tomato/strawberry: pays out on a schedule (4 payouts), then decays to a weed |
| **Tranche** | a bounded slice of a sell order, spread over turns so you don't choke the stomach |
| **Blocked PLANT** | order more `PLANT X` than seeds you hold and **all** of them fail that turn |
| **The ladder** | Kaggle's Bradley-Terry rating — like a chess club Elo: beat stronger farms, climb; scores wobble game to game, so judge the *trend*, not one number |

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.32.2"

In [ ]:
# Chapter 0.5 — Meet the farm you inherited.
#
# kaggle-environments ships the whole town: the board, the market, the
# shops, and the built-in rivals ("random", "starter", ...). Let's open
# the gate and look around.

from kaggle_environments import make

env = make("kaggriculture", debug=True)
print(f"Environment : {env.name} v{env.version}")
print(f"Season      : {env.configuration.episodeSteps} turns "
      f"({env.configuration.episodeSteps // env.configuration.turnsPerDay} days x "
      f"{env.configuration.turnsPerDay} turns/day)")

# Play a quick noise game (two wanderers) and inspect what a farmer sees:
env.run(["random", "random"])
obs = env.steps[1][0].observation            # step 1 = the first decision point

print("\n--- your first observation ---")
print(f"player       : {obs.get('player')}")
print(f"day, hour    : {obs.get('day')}, {obs.get('hour')}")
farm = obs.get("farms")[obs.get("player")]
print(f"your money   : ${farm.get('money'):.0f}   (the coffee tin from the letter)")
print(f"your quadrants: {farm.get('unlocked_quadrants')}  (NW is free, others: $1k/$2k/$4k)")
print(f"farmer at    : {farm.get('farmer')}   (the barn/shed sits at the board center)")

prices = obs.get("market").get("prices")
print("\n--- today's market (the town's price board) ---")
for item, price in prices.items():
    print(f"  {item:<11s} ${price}")

## 🍈 Season 1 — The Melon Dream

**You plant your first seed the way Uncle Robert did fifty years ago:** melons. One crop, one farmer, one rhythm — buy a seed, water it, harvest it, sell when the price is right. The neighbor's kid (Kaggle's own bovard, in his official getting-started notebook) wrote this exact farmer, and we've rewritten him cleanly here:

1. Keep exactly **one melon seed** stocked — buy when out.
2. Walk to whatever melon needs watering or harvesting; plant on empty tiles.
3. Sell the entire barn **only when melon ≥ $200**.
4. Repeat for 720 turns.

**What to watch in the cell below:** the season ends with roughly **$6,000 in the bank vs $0** — proof that the simple *observe → decide → act* loop already buries a bot that wanders aimlessly. This is your floor. Every farmer after this one only gets richer.

> **But the Melon Dreamer has four flaws, and the Almanac explains every one of them:**
>
> | # | The flaw | The Almanac's law it breaks | Fixed in |
> |---|---|---|---|
> | 1 | One farmer, one plot, no hires | Law 1 — time is the real currency | Season 3 |
> | 2 | 100% melon portfolio | Law 3 — the melon stomach never digests | Season 2 |
> | 3 | Never fertilizes | Law 2½ — water-days are money | Season 3 |
> | 4 | Dumps the whole barn in one order | Law 3 — you choke your own stomach | Season 2 |
>
> Notice flaw 4 especially: the Dreamer refuses to sell below $200, then sells *everything at once* — which itself drags the price below $200 for the next cycle. He keeps tripping over his own threshold. It's the most human mistake in the game.

In [ ]:
"""AGENT 1 — Melon Maxxer (baseline, faithful clean rewrite of bovard's starter).

Strategy: grow one thing, sell it above a threshold, repeat.
Known flaws (fixed one-by-one by the later agents):
  1. Never hires hands or buys land  -> capped at 1 farmer / 25 tiles
  2. Single crop                     -> no fallback when melon price crashes
  3. Never fertilizes                -> leaves yield on the table
  4. Dumps entire stock in one order -> crashes the price mid-sale
"""
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS

MELON_SEED_COST = CROPS["MELON"]["seed"]                # $80
MELON_MAX_YIELD_DAY = CROPS["MELON"]["max_yield_day"]   # melon stops growing day 12
SELL_THRESHOLD = 200


def _step_toward(fx, fy, tx, ty):
    """One tile of Manhattan movement (y grows DOWNWARD on this board)."""
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None


def _find_target_tile(farm, board_size, have_seed):
    """Nearest melon tile needing harvest/water, else nearest empty tile."""
    fx, fy = farm["farmer"]
    candidates = []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile["crop"] == "MELON":
                purpose = None
                if tile["yield_units"] > 0:
                    purpose = "harvest"
                if not tile["watered_today"]:
                    purpose = "water" if purpose is None else purpose
                if purpose:
                    candidates.append((x, y, purpose))
            elif tile is None and have_seed:
                candidates.append((x, y, "plant"))
    if not candidates:
        return None
    priority = {"harvest": 0, "water": 1, "plant": 2}
    candidates.sort(key=lambda c: (priority[c[2]], abs(c[0] - fx) + abs(c[1] - fy)))
    return candidates[0]


def melon_maxxer(obs):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    fx, fy = farm["farmer"]
    tile = farm["tiles"][fy][fx]
    day = obs.get("day", 0)

    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})
    melon_price = market_prices.get("MELON", 0)

    market = []
    # Market rule: only sell when the price clears our threshold.
    melons_in_shed = shed.get("MELON", 0)
    if melons_in_shed > 0 and melon_price >= SELL_THRESHOLD:
        market.append(["SELL", "MELON", melons_in_shed])  # flaw 4: all at once
    # Keep exactly one melon seed stocked (flaw 1: never scales up).
    if seeds.get("MELON", 0) == 0 and farm["money"] >= MELON_SEED_COST:
        market.append(["BUY_SEED", "MELON", 1])

    farmer = ["PASS"]
    if isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile["crop"] == "MELON":
        age = day - tile["planted_day"]
        if age >= MELON_MAX_YIELD_DAY and tile["yield_units"] > 0:
            farmer = ["HARVEST"]
        elif not tile["watered_today"]:
            farmer = ["WATER"]
        else:
            target = _find_target_tile(farm, board_size, seeds.get("MELON", 0) > 0)
            if target:
                step = _step_toward(fx, fy, target[0], target[1])
                if step:
                    farmer = [step]
    elif tile is None and seeds.get("MELON", 0) > 0:
        farmer = ["PLANT", "MELON"]
    else:
        target = _find_target_tile(farm, board_size, seeds.get("MELON", 0) > 0)
        if target:
            step = _step_toward(fx, fy, target[0], target[1])
            if step:
                farmer = [step]

    return {"farmer": farmer, "hands": [], "market": market}

In [ ]:
# Season 1, harvest day: The Melon Dreamer vs the wandering random bot.
env = make("kaggriculture", debug=True)
env.run([melon_maxxer, "random"])

for i, s in enumerate(env.steps[-1]):
    print(f"Player {i}: bank = ${s.reward:,.0f}   status = {s.status}")

# Want to *watch* the season play out? Uncomment on Kaggle for the replay UI:
# env.render(mode="ipython", width=800, height=600)

## 🥕 Season 2 — The Market Lesson

**Winter comes early the second year.** Your melons flood the town's stomach and the price of a fruit that sold for $250 on day one sags toward $50 while your fields sit empty. So in Season 2 you stop being a *price-taker* and become a *price reader*.

**The one formula that changes everything** — computed fresh, every single turn, from the market board:

```text
score(crop) = (typical_yield × today's price − seed cost) / cycle days
```

Melon at $250 scores ~$129/tile-day and deserves the field. Melon at $50 scores ~$20 — suddenly carrots and wheat are the better employees. The farm *self-balances* because the score always uses live prices, and a hard 65% cap stops any single crop from owning the whole portfolio.

**The five upgrades, as a story of habits:**

| Upgrade | The habit behind it |
|---|---|
| Multi-crop portfolio | never marry one crop; re-hire your tiles every cycle based on today's wages (prices) |
| Tranche selling | sell in **baskets of a few units** so the town's stomach digests between bites — dips genuinely recover for digestible goods |
| Price floors | hold stock until the price is worth it (65% of base) — except in the final panic |
| Harvest at peak | pick one-time crops the moment `yield_units` caps (the day-10 melon trick) |
| Hygiene | dig weeds, water fresh plantings same-day, never plant what can't ripen by the buzzer |

**What to watch below:** Agent 2 banks **~4× Agent 1's score** and beats the Melon Dreamer head-to-head — diversification isn't just safer, it's richer.

One structural secret before the code: harvested goods ride in a worker's **pockets** and only reach the barn (where selling happens) at dusk or on a DROP — so *today's* harvest is sold *today evening or tomorrow morning*. Good farmers plan around that lag.

In [ ]:
"""AGENT 2 — Diversified Farmer.

Upgrades over the Melon Maxxer:
  1. Multi-crop portfolio, re-scored every turn with TODAY'S prices
     (profit per tile-day = (typical_yield x price - seed_cost) / cycle_days)
  2. Tranche selling — never dumps: caps units sold per product per turn so
     we ride the price down gently instead of nuking it
  3. Price floors — hold stock unless the market pays >= FLOOR_FRAC x base
     (except the final dump — unsold goods are worth $0 at the buzzer)
  4. Correct harvest timing — pick one-time crops the moment they are full
     (a watered melon caps out at day 10, two days before the baseline picks)
  5. Weed control + end-game discipline (stop planting, liquidate everything)
"""

# --- Hardcoded game tables (verified against kaggriculture.py; keeps this file portable) ---
CROP_META = {
    #            seed$  base$  max_yld peak_day cycle_days typical_yield(unfertilized, watered)
    "WHEAT":      {"seed": 10,  "base": 25,  "max_yield": 6, "peak": 4,  "cycle": 5,  "typ": 4},
    "CARROT":     {"seed": 20,  "base": 35,  "max_yield": 4, "peak": 3,  "cycle": 4,  "typ": 3},
    "TOMATO":     {"seed": 50,  "base": 60,  "max_yield": 4, "peak": 8,  "cycle": 12, "typ": 4},
    "STRAWBERRY": {"seed": 100, "base": 120, "max_yield": 4, "peak": 10, "cycle": 17, "typ": 4},
    "MELON":      {"seed": 80,  "base": 250, "max_yield": 6, "peak": 10, "cycle": 11, "typ": 6},
}
BASE_PRICES = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
               "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100}
PRODUCTS = list(BASE_PRICES)

SEASON_DAYS = 30
FLOOR_FRAC = 0.65          # sell only when price >= 65% of base
MELON_PORTFOLIO_CAP = 0.6  # never let one crop fill more than 60% of tiles
TRANCHE = {                # max units sold per product per turn
    "MELON": 4, "WOOL": 3, "MILK": 5, "STRAWBERRY": 5, "TOMATO": 10,
    "CARROT": 15, "EGG": 10, "WHEAT": 25, "FERTILIZER": 3,
}
LAST_PLANT_DAY = {"WHEAT": 26, "CARROT": 26, "TOMATO": 19, "STRAWBERRY": 14, "MELON": 19}
SEED_BUFFER = 4            # spare seeds to keep on hand per crop
ONGOING = ("TOMATO", "STRAWBERRY")


def _step_toward(fx, fy, tx, ty):
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None


def _crop_score(crop, price):
    """Expected profit per tile-day at the CURRENT market price."""
    m = CROP_META[crop]
    return (m["typ"] * price - m["seed"]) / m["cycle"]


def _best_seeded_crop(seeds, prices, day):
    """Highest-scoring crop we can still plant in time and actually hold seeds for."""
    options = [c for c, n in seeds.items() if n > 0 and day <= LAST_PLANT_DAY[c]]
    return max(options, key=lambda c: _crop_score(c, prices[c]), default=None)


def _plan_seed_purchases(farm, seeds, prices, day, money):
    """Rank crops by profit-per-tile-day, buy seeds for the best affordable ones."""
    orders = []
    melon_planted = sum(
        1 for row in farm["tiles"] for t in row
        if isinstance(t, dict) and t.get("kind") == "PLANT" and t["crop"] == "MELON")
    tiles = len(farm["tiles"]) ** 2
    empty = sum(1 for row in farm["tiles"] for t in row if t is None)
    for crop in sorted(CROP_META, key=lambda c: _crop_score(c, prices[c]), reverse=True):
        if empty <= 0:
            break
        if day > LAST_PLANT_DAY[crop]:
            continue
        if crop == "MELON" and melon_planted > MELON_PORTFOLIO_CAP * tiles:
            continue  # portfolio cap: too much melon = glut risk
        want = min(SEED_BUFFER, max(1, empty))
        if seeds.get(crop, 0) < want:
            n = want - seeds.get(crop, 0)
            cost = n * CROP_META[crop]["seed"]
            if money - cost >= 40:  # keep a small liquidity reserve
                orders.append(["BUY_SEED", crop, n])
                money -= cost
                empty -= n
    return orders


def diversified_farmer(obs):
    try:
        farms = obs.get("farms", [])
        player = obs.get("player", 0)
        private = obs.get("private", {}) or {}
        if not farms or player >= len(farms):
            return {"farmer": ["PASS"], "hands": [], "market": []}
        farm = farms[player]
        board = len(farm["tiles"])
        fx, fy = farm["farmer"]
        tile = farm["tiles"][fy][fx]
        day = obs.get("day", 0)
        hour = obs.get("hour", 0)
        seeds = private.get("seeds", {})
        shed = private.get("shed", {})
        prices = (obs.get("market", {}) or {}).get("prices", {})

        # ---------------- MARKET ----------------
        market = []
        # 1) Sell in tranches above a price floor; dump everything at the buzzer.
        for product in PRODUCTS:
            stock = shed.get(product, 0)
            if stock <= 0:
                continue
            price = prices.get(product, 0)
            if day >= SEASON_DAYS - 2 or price >= FLOOR_FRAC * BASE_PRICES[product]:
                market.append(["SELL", product, min(stock, TRANCHE.get(product, 5))])
        # 2) Restock seeds for the best-scoring crops.
        if day < SEASON_DAYS - 4:
            market += _plan_seed_purchases(farm, seeds, prices, day, farm["money"])

        # ---------------- FARMER ----------------
        # Scan the whole farm and build a priority task list:
        #   0 = harvest ripe, 1 = water thirsty, 2 = plant empty, 3 = dig weeds
        tasks = []
        for y in range(board):
            for x in range(board):
                t = farm["tiles"][y][x]
                if isinstance(t, dict) and t.get("kind") == "PLANT":
                    meta = CROP_META[t["crop"]]
                    age = day - t["planted_day"]
                    ripe = (t["crop"] in ONGOING and t["yield_units"] > 0) or \
                           (t["crop"] not in ONGOING and
                            (t["yield_units"] >= meta["max_yield"] or age >= meta["peak"]))
                    if ripe and t["yield_units"] > 0:
                        tasks.append((0, x, y))
                    elif not t["watered_today"]:
                        tasks.append((1, x, y))
                elif isinstance(t, dict) and t.get("kind") == "WEED":
                    tasks.append((3, x, y))
                elif t is None and hour <= 21:
                    if _best_seeded_crop(seeds, prices, day):
                        tasks.append((2, x, y))

        # Act on the tile we are standing on first (free action, no walking).
        farmer = ["PASS"]
        if isinstance(tile, dict) and tile.get("kind") == "PLANT":
            meta = CROP_META[tile["crop"]]
            age = day - tile["planted_day"]
            ripe = (tile["crop"] in ONGOING and tile["yield_units"] > 0) or \
                   (tile["crop"] not in ONGOING and
                    (tile["yield_units"] >= meta["max_yield"] or age >= meta["peak"]))
            if ripe and tile["yield_units"] > 0:
                farmer = ["HARVEST"]
            elif not tile["watered_today"]:
                farmer = ["WATER"]
        elif isinstance(tile, dict) and tile.get("kind") == "WEED":
            farmer = ["DIG"]
        elif tile is None and hour <= 21:
            best = _best_seeded_crop(seeds, prices, day)
            if best:
                farmer = ["PLANT", best]

        # Otherwise walk toward the most urgent nearby task.
        if farmer == ["PASS"] and tasks:
            tasks.sort(key=lambda t: (t[0], abs(t[1] - fx) + abs(t[2] - fy)))
            step = _step_toward(fx, fy, tasks[0][1], tasks[0][2])
            if step:
                farmer = [step]
        return {"farmer": farmer, "hands": [], "market": market}
    except Exception:
        # Grandmaster rule #1: a crashed agent scores zero. Never crash.
        return {"farmer": ["PASS"], "hands": [], "market": []}

In [ ]:
# Season 2, harvest day: the Market Gardener takes the same fields.
env = make("kaggriculture", debug=True)
env.run([diversified_farmer, "random"])
print("A2 vs random :", f"${env.steps[-1][0].reward:,.0f}", "vs",
      f"${env.steps[-1][1].reward:,.0f}")

env = make("kaggriculture", debug=True)
env.run([diversified_farmer, melon_maxxer])
print("A2 vs A1     :", f"${env.steps[-1][0].reward:,.0f}", "vs",
      f"${env.steps[-1][1].reward:,.0f}", " (the Season-1 farmer loses)")

## 🏭 Season 3 — The Empire (Farm OS)

**By year three you stop gardening and start running a company.** You buy the two neighboring plots, hire half the village, and — the Almanac's best-kept secret — you discover that **geese are printing machines**: a $300 goose lays ~2 eggs a day for one wheat of feed on a market that *never* crashes, and every bird on the farm drops a free jug of fertilizer each morning, which doubles every crop's watering bonus (Law 2½). This is the season the notebook is named after: **Farm OS**, a farmhouse operating system that re-plans the entire business **from scratch every single turn** — no memory, no stale habits, just a fresh audit of what it sees. That *stateless re-planning* pattern is what makes competition agents debuggable and robust.

### The eight engines

```
                 ┌──────────────────────────────────────────────┐
                 │            FARM OS  ·  GOOSE EMPIRE          │
                 ├───────────────┬──────────────┬───────────────┤
                 │  RACE (melon) │ SCOUT (rival)│  LAND         │
                 │ sell instantly│ read their   │ buy quadrant  │
                 │ no floor, ever│ standing crop│ when utilized │
                 │ budget tiles  │ pivot if they│ ($1k/$2k/$4k) │
                 │ vs headroom   │ flood melon  │               │
                 ├───────────────┼──────────────┴───────────────┤
                 │  LABOR        │  CROPS          │  HERD      │
                 │ re-hire every │ fert at window  │ geese: eggs│
                 │ morning, 12   │ openers (+2/day)│ + free fert│
                 │ hands, zones  │ full-yield melon│ cows/sheep │
                 ├───────────────┴─────────────────┴────────────┤
                 │  PURSE: spend projector (never bank at $0)   │
                 ├──────────────────────────────────────────────┤
                 │                MARKET                        │
                 │  sell ALL now (the telescoping law)          │
                 │  keep only feed wheat + window fert          │
                 │  hot-drop cargo | guaranteed final-day sweep │
                 └──────────────────────────────────────────────┘
```

**The RACE engine — winning the melon race.** Melon demand is a *finite, non-refillable resource* worth ~$26K all season, and both farms sell into the **same** curve — the environment interleaves both players' sells unit-by-unit at one shared price. So the race is decided by *who brings melons to town on the earlier day*. Farm OS sells every melon the moment it reaches the barn (one order, no floor), **budgets** plantings against the remaining headroom — counting *the rival's standing melons, read straight from the public farm board* — and if the market dies early, it **digs up its young melons and replants** something the town still wants. A sunk melon is worth less than a live crop.

**The SCOUT engine.** `obs["farms"]` holds *both* farms — your rival's tiles, money and hands are public. One scan per turn predicts their dump; that one number decides whether wave-2 melons get planted at all.

**The PURSE engine — the $0 lesson.** (We lost a whole season learning this one.) A farm that spends its last coin cannot hire a single hand in the morning — and a farm with no hands turns to weeds in two days, geese starve, and income stops forever. So every purchase this turn must pass a **spend projector**: the projected bank after *all* of today's buys never dips below a working-cash floor. Ambition buys land and geese only when the purse stays alive.

**Land & Labor.** A quadrant pays for itself in days *if* hands exist to work it. Hands vanish nightly, so the farm re-hires every morning — 12 hands cost ~$376/day in fib wages and one melon tile pays that back; **never** gate hiring on cash (the env refuses unaffordable hires on its own). Dispatch runs off **one priority task board**, each task claiming the nearest free worker in its own **territory zone** so the day doesn't evaporate in walking.

**The FERTILIZE protocol.** Because fertilizer doubles *watering-day* yield, timing is everything: melons get jugged **exactly on day 6** (window opener → 6 units by day 8, harvested full-size on day 10), carrots and wheat on day 2, tomatoes across their payout days 8–11, strawberries across 10–16. Geese supply the jugs for free.

**Full-yield melons.** The old farmer harvested melons at day 10 with ~4–5 units. Farm OS waters through **day 11** and harvests at the full 6 units — but only when the headroom math says the extra day doesn't lose the race. Yield vs speed, decided by data, per game.

**The carrier rule — the bug that starved the geese.** (Season 3's second expensive lesson.) `FEED` and `FERTILIZE` consume items from the **worker's own pockets**, not the barn. Early versions assigned chores to the *nearest* worker — whose pockets were empty — so geese starved to death beside a full wheat shed. Farm OS now runs **carrier missions**: a worker picks up wheat/fertilizer at the barn, then delivers to each animal/field, nearest-first. Feeding is the farm's #1 morning chore — an animal unfed for two days simply flies away with your $300.

**The MARKET engine — Season 4, when the Empress fired her own rule.** Here is the notebook's favorite plot twist. Season 2 taught *baskets and floor prices*, and the Almanac's Law 3 made it sound like wisdom. Then the Empress did what every grandmaster does: **she re-ran the math on her own habits.** Selling S units of anything slides the price down a curve — and the total revenue of those S units is **the same sum no matter how you slice it** (sell 10 today and 10 tomorrow, or all 20 today: telescoping, the revenue adds up identically). Waiting literally earns **$0**. Worse, two real forces punish waiting: the rival's supply is *coming* (units sold early grab the high end of the shared curve), and the barn's 100 shelves overflow at dusk (overflow is *discarded*). So the v4 law is: **sell everything the moment it lands** — one order per product, no floors, no tranches. The only stock kept back is operational: a wheat feed buffer for the animals and fertilizer jugs for the yield windows. And early cash compounds: wages, land and geese bought a day sooner pay back a day sooner. (We measured this locally before shipping: the new rule beats the old farmer 5–0.)

**The v4 audit, in one table:**

| Upgrade | Why it pays | Verdict |
|---|---|---|
| 🗑️ Floors & tranches removed | waiting earns $0 (telescoping); rival supply + dusk overflow punish holding | ✅ shipped, 5–0 vs v3 |
| 🧪 Fertilizer triage | free goose jugs routed to melon window openers first, then wheat/carrot doublers | ✅ shipped |
| 🐐 Flock 9 → 12 | payback ~5–6 days on a deep egg market + 1 free jug/day each | ✅ shipped |
| 🗺️ Land window 18 → 20 | the $4,000 quadrant still repays by day 20 when cash is flush | ✅ shipped |
| 🌾 Wheat-for-profit rows, 14-hand crews, aggressive melon waves, a 'honest units/day' crop scorer | promising ideas… | ❌ measured **worse** locally, rejected |

That last row is the real grandmaster skill: **ideas are cheap, tournaments are truth.** Every ❌ above looked smart on paper and lost on the field.

### Season 5 — The $100 free lunch (and the five ideas that failed)

**The most valuable hour of the Empire's life was spent reading its own receipts.** A season-end audit of the v4 farm — replayed turn by turn — found something absurd: every goose drops **one free jug of fertilizer every single morning**, and the town *re-arms that gift every night no matter what* — even the env's own code does it unconditionally. Fertilizer sells for ~$100. Twelve geese = **$1,200/day of free product**. And the audit showed the old farm had collected… **32 jugs. All season.** A fortune was lying in the barnyard, swept around daily by workers who never had a spare action.

So the v5 farm builds a real **fertilizer pipeline**: collecting jugs becomes a scheduled chore at its own priority — *after* animal harvests (a ripe egg beats a jug) and *before* care (care can wait an hour; a melon sale cannot wait at all) — plus a new carrying rule: pockets holding **4+ jugs detour to the barn**. The pipeline is opponent-agnostic (it works the same against any rival, seen or unseen — that's what makes it safe for *both* leaderboard boards), and it paid: **5–0 vs the v4 Empress head-to-head, +$1.3–1.6K average bank, with every robustness gate still green.**

**But the honest number is bigger than the shipped code — because four other "obvious" upgrades measured WORSE, and keeping them out is worth as much as shipping the one that worked:**

| v5 idea | The tempting logic | What the tournament said | Verdict |
|---|---|---|---|
| 🚰 **Fertilizer pipeline** | 12 free jugs/day ≈ $1,200/day ignored | **5–0 vs v4**, +$1.3–1.6K | ✅ shipped |
| 🌱 Flow planting (fill 60–94 idle tiles with carrots/wheat all season) | idle land is free money | **−$6–8K** — the extra watering starved the melon race | ❌ rejected |
| 🐄 3 cows + 3 sheep (milk sold at $208–386!) | scarcity premiums are real | herd chores displaced watering; **lost** | ❌ rejected |
| 🔓 "Unlock the empire" (buy geese through the day-5 cash desert) | a goose pays back in 5 days | flock grew 1→9 as designed… but feed-buying **raised the rival's wheat price** through the shared market: 3–2 | ❌ rejected |
| 🥇 Bump CARE/harvest priority | an uncared goose lays 1 egg, a cared one 2 | **−$5K** — care stole watering labor | ❌ rejected |

> **Law 5 — Labor is zero-sum.** Every tile, animal and chore draws from the *same* 288 daily actions. A task is not worth its revenue; it is worth its revenue *minus the chore it displaces*. The idle tiles were never free — they were a trap. This is why the v5 farm ships one pipeline and refuses four shiny ideas.
>
> And one quiet bug-fix rides along: the env's milk curve is linear (the old table said square), and the final-day fertilizer sweep no longer collects what it could never sell — the barn now empties **clean at the buzzer**.

**What to watch below:** the Empress banks **$40–45K** vs the $6K of Season 1, beat every previous farmer and every built-in rival across two independent seed batches — including 12–0 sweeps of `random` and `starter` on seeds it had *never seen* — and the *mirror match stays healthy*: two copies of Farm OS read the same finite melon curve and quietly split it, instead of jointly strangling it.

In [ ]:
%%writefile submission.py
"""AGENT 3 - Farm OS 5.0 "Goose Empire v5" (grandmaster full-economy agent).

Re-planned from scratch EVERY turn (stateless: no stale state, no drift).
v5 = v4 + ONE shipped, locally-validated upgrade - and five measured lessons
(~280 tournament episodes across 5 bisect rounds; every idea below was
implemented, measured against v4 head-to-head, and either shipped or rejected
on data, not taste):

  SHIPPED - FERTILIZER PIPELINE (FERT_PIPELINE): every goose donates 1 free
  fertilizer per day (the env re-arms the flag nightly no matter what). v4
  collected ~32 units ALL SEASON at $75-100 each. The v5 board collects at
  priority 3.45 (after animal harvests, before care - it must never delay a
  melon sale or a survival watering) and hot-drops pockets holding >=4.
  Fertilizer has ZERO town consumption, so its curve only softens with our
  own supply (linear, $0.20/unit) - the deepest market on the map.
  Measured: 5-0 vs v4 head-to-head and vs v3, +$1.3-1.6K average.

  MEASURED-AND-REJECTED (kept here as lessons, not code paths):
  - FLOW PLANTING (fill the 60-94 idle tiles with carrot/wheat all season):
    LOST $6-8K. Labor is zero-sum: the extra watering/harvest tasks starved
    the melon race and the strawberry engine. The idle tiles are not free -
    they are a labor trap.
  - SCARCITY LIVESTOCK (3 cows, 3 sheep, geese 14): sheep cost 4 task-
    actions/day for ~$100/day of wool; the herd displaced watering. LOST.
  - 'UNLOCK THE EMPIRE' (goose gate $1000->$500, wage-scaled spend floor):
    grew the flock 1-3 -> 7-9 birds as designed... and bought feed wheat at
    $46-55, which SUBSIDIZES the opponent's wheat sales through the shared
    market. Head-to-head fell to 3-2. Wheat self-sufficiency (10 tiles)
    displaced melons instead: 1-4. The poverty desert is a FEATURE - it
    protects the melon race's labor.
  - CARE/ANIMAL-HARVEST priority bumps (2.9/2.95): LOST $5K. An uncared
    goose costs an egg; a delayed melon watering costs a crop. v4's board
    order is co-adapted and sacred.
  - KEEP_FERT 4, GOOSE 14, MELON_CAP 150, LAND_DAY 22, HAND_CAP 14: all
    noise or regression on 5-seed paired tests.

  The v4 laws all carried forward unchanged (each was itself a measured
  winner once): INSTANT-SELL, FERT TRIAGE, SPEND PROJECTOR, RACE/SCOUT,
  staged seed budgets, hot-drop cargo, final-day sweep.

  RACE   - melons are a NON-RENEWABLE market: the town center is the only
           buyer (~1 unit/day), so a glut never recovers. Sell every melon
           the moment it lands in the shed, and budget plantings against
           the remaining price headroom (opponent's standing melons count).
  SCOUT  - the opponent's farm is PUBLIC. Their standing young melons tell
           us the dump that is coming; overshoot -> dig-and-pivot.
  LAND   - buy the next quadrant when cash clears cost + reserve.
  LABOR  - hands re-hired every morning (fib wages are noise). Scale to 12;
           territory zones so nobody burns the day walking.
  CROPS  - the WATER action itself pays +1 yield per watering inside a
           crop's yield window (+2 if fertilized that day). Fertilizer is a
           yield MULTIPLIER, not a growth potion.
  HERD   - a cared+fed goose lays 2 eggs/day on a DEEP log market, plus 1
           free fertilizer/day - the fuel for the FERTILIZER PIPELINE.
  MARKET - melon: one order, dump everything, always. Eggs: deep market.
           Hard liquidation from day 28 + guaranteed final-day sweep:
           unsold goods score ZERO.
"""

import math

# ---------------- Game tables (mirror kaggriculture.py; keeps file portable) ----------------
CROP_META = {
    "WHEAT":      {"seed": 10,  "first": 2,  "peak": 4,  "max": 6, "ongoing": False, "interval": 0},
    "CARROT":     {"seed": 20,  "first": 2,  "peak": 3,  "max": 4, "ongoing": False, "interval": 0},
    "TOMATO":     {"seed": 50,  "first": 8,  "peak": 8,  "max": 4, "ongoing": True,  "interval": 1},
    "STRAWBERRY": {"seed": 100, "first": 10, "peak": 10, "max": 4, "ongoing": True,  "interval": 2},
    "MELON":      {"seed": 80,  "first": 10, "peak": 12, "max": 6, "ongoing": False, "interval": 0},
}
BASE = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120, "MELON": 250,
        "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100}
PRODUCTS = list(BASE)

# Exact 'glut side' curves from the env (inventory above I0=10000).
# v5 fix: MILK above-curve is LINEAR in the env (v3/v4 said sqrt; unused by
# decisions but wrong on paper - zero-mistake edition).
MK = {
    "WHEAT":      {"T": 400, "f": "log",    "tgt": 0.20},
    "CARROT":     {"T": 450, "f": "sqrt",   "tgt": 0.70},
    "TOMATO":     {"T": 200, "f": "sqrt",   "tgt": 0.60},
    "STRAWBERRY": {"T": 100, "f": "linear", "tgt": 1.60},
    "MELON":      {"T": 300, "f": "sq",     "tgt": 3.60},
    "EGG":        {"T": 332, "f": "log",    "tgt": 0.20},
    "MILK":       {"T": 122, "f": "linear", "tgt": 1.60},
    "WOOL":       {"T": 105, "f": "sq",     "tgt": 3.20},
    "FERTILIZER": {"T": 200, "f": "linear", "tgt": 0.40},
}
I0 = 10000


def _shape(f, x):
    x = max(0.0, float(x))
    if f == "linear": return x
    if f == "sq":     return x * x
    if f == "sqrt":   return math.sqrt(x)
    if f == "log":    return math.log(1.0 + x)
    return x


def price_above(item, gap):
    """Env-exact price when market inventory is I0 + gap (gap >= 0)."""
    p = MK[item]
    amp = p["tgt"] * BASE[item] / _shape(p["f"], p["T"])
    return max(1, int(round(BASE[item] - amp * _shape(p["f"], gap))))


SEASON_DAYS = 30
RESERVE = 600                       # cash we never spend (seeds + wages buffer)
SPEND_FLOOR = 400                   # hard floor: after ALL buys this turn, money
                                    # must stay above this (~1 day of wages).
SHED_TILES = [(4, 4), (5, 4), (4, 5), (5, 5)]

# v4 LAW: floors and tranches are GONE. Proof: revenue from selling S units
# telescopes to the same sum no matter how you slice it (the curve only moves
# with cumulative supply), so timing earns exactly $0 - while the opponent's
# incoming supply and end-of-day shed overflow both punish waiting. Sell
# everything the moment it lands; the only keeps are operational buffers.
FLOOR = {k: 0.0 for k in ("MELON", "STRAWBERRY", "MILK", "WOOL",
                          "TOMATO", "EGG", "WHEAT", "CARROT", "FERTILIZER")}
TRANCHE = {k: 999 for k in FLOOR}
WOOL_DUMP_DAY = 24                  # sq-curve market: never bank wool late

MELON_TARGET = 12                   # wave-1 tile budget: land beats melon volume
MELON_WAVE2 = 10                    # wave-2 budget (sells into a partly-used curve)
MELON_HEADROOM_CAP = 135            # projected gap above which marginal melon < ~$68
MELON_LAST_PLANT = 15

# ---------------- v5 feature flags (ship state: only the measured winner ON) ----------------
FLOW_PLANT   = False  # REJECTED by bisect: -$6-8K (labor zero-sum; see docstring)
FLOW_DAILY_CAP = 8    # (inactive with FLOW_PLANT off)
HAND_CAP     = 12     # 14 measured noise-to-negative on paired seeds
FERT_PIPELINE = True  # SHIPPED: 5-0 vs v4 head-to-head, +$1.3-1.6K avg
KEEP_FERT    = 6      # 4 measured as noise on top of the pipeline
LIVESTOCK    = False  # REJECTED: sheep/cow expansion lost to task displacement
GOOSE_BUDGET, GOOSE_FIRST, GOOSE_LAST = 12, 1, 14
COW_BUDGET, COW_FIRST, COW_LAST = 2, 5, 10
SHEEP_BUDGET, SHEEP_FIRST, SHEEP_LAST = 2, 6, 11
if LIVESTOCK:
    COW_BUDGET, SHEEP_BUDGET = 3, 2   # (inactive with LIVESTOCK off)
FEED_GATE   = 42 if not LIVESTOCK else 60
FEED_CAP    = 12 if not LIVESTOCK else 24
DEAD_PIVOT  = False  # REJECTED: never binds before the market self-corrects
OPP_RIPE    = False  # REJECTED: rarely binds (identical to parity on 5 seeds)
CARE_PRIO   = 3.5    # bisect: 2.9 REGRESSED (-$5K) - care must not outrank
                     # watering/harvest; v4's co-adapted board wins.
ANIMAL_HARVEST_PRIO = 3.3  # bisect: 2.95 regressed too. Board order is sacred.
WHEAT_TARGET_UNLOCK = 8    # (inactive: UNLOCK rejected - feed buying subsidizes
FEED_GATE_UNLOCK = 55      #  the opponent's wheat sales via the shared market)
UNLOCK      = False  # REJECTED: flock grew 1-3 -> 7-9 as designed, but feed
                     # demand raised the opponent's wheat price; 3-2 vs v4.
POCKET_JOB  = False  # REJECTED by A/B ($31.3K vs $35.0K, W2-L3): making the
                     # delivery carrier walk the full trip costs more labor than
                     # the pickup/drop bounce it replaces. The bounce IS the
                     # equilibrium; shed-leftover birds at the buzzer are rare
                     # sunk cost (buys outran builds), not unsold product.
ORDER_FULL  = True   # v4 order semantics: sells top-5 + hires up to 3; the
                     # 10-order cap is absorbed by DEFERRING seed/land/animal
                     # buys to roomy turns (never the other way round).
                     # (bisect: top-3 sells STARVED the shed; 1 hire/turn broke
                     # the day-1 bootstrap - sells & hires never wait.)
LAST_SEED_DAY = {"WHEAT": 24, "CARROT": 25, "TOMATO": 18, "STRAWBERRY": 16, "MELON": MELON_LAST_PLANT}
LAND_LAST_DAY = 20                  # v4: SE quadrant still pays back by day 20

_ERR = {"n": 0}   # bounded crash logging: never spam, never crash silently


def _step_toward(fx, fy, tx, ty):
    if fx > tx: return "WEST"
    if fx < tx: return "EAST"
    if fy > ty: return "NORTH"
    if fy < ty: return "SOUTH"
    return None


def _quadrant(x, y):
    return ("N" if y < 5 else "S") + ("W" if x < 5 else "E")


def _at_shed(pos):
    return tuple(pos) in SHED_TILES


def _hand_target(plants, animals, day, money, last_day):
    """~1 action/tile/day for watering + harvest/plant/animal churn.
    Twelve hands = $376/day in fib wages; one melon tile pays that back."""
    t = 2 + (plants + animals) // 9
    if day >= 1 and money >= 100:  t = max(t, 3)
    if day >= 3 and money >= 500:  t = max(t, 5)
    if day >= 6:                   t = max(t, 7)
    if day >= 8:                   t = max(t, 8)
    if day >= 12:                  t = max(t, 10)
    if plants >= 70 and HAND_CAP >= 14: t = max(t, 13)   # flow era needs runners
    if last_day:                   t = max(t + 3, 6)   # final day: extra runners
    return max(2, min(HAND_CAP, t))


def _wage_bill(n):
    """Daily fib wage bill for n hands: fib(1)+...+fib(n)."""
    a, b, s = 1, 1, 0
    for _ in range(max(0, n)):
        s += a
        a, b = b, a + b
    return s


def farm_os(obs):
    try:
        farms = obs.get("farms", [])
        player = obs.get("player", 0)
        private = obs.get("private", {}) or {}
        if not farms or player >= len(farms):
            return {"farmer": ["PASS"], "hands": [], "market": []}
        farm = farms[player]
        board = len(farm["tiles"])
        day = obs.get("day", 0)
        hour = obs.get("hour", 0)
        seeds = dict(private.get("seeds", {}))
        shed = dict(private.get("shed", {}))
        market = obs.get("market", {}) or {}
        prices = market.get("prices", {}) or {}
        inv = market.get("inventory", {}) or {}
        money = farm["money"]
        unlocked = len(farm["unlocked_quadrants"])
        last_day = day >= SEASON_DAYS - 1            # day 29: last chances
        sellout = day >= SEASON_DAYS - 2             # day 28+: liquidate mode

        # ------------------ SCAN OUR FARM ------------------
        plants = 0
        crop_count = {"MELON": 0, "WHEAT": 0, "CARROT": 0, "TOMATO": 0, "STRAWBERRY": 0}
        thirsty_ongoing, thirsty_melon_win, thirsty_other = [], [], []
        harvest_melon, harvest_fast, harvest_ongoing, harvest_animal = [], [], [], []
        fert_melon, fert_fast, fert_ongoing, weeds, empty, dig_pivot = [], [], [], [], [], []
        dead_ongoing = []                            # v5: dead-market pivot candidates
        animals, coops_empty, pastures_empty = [], [], []
        geese = cows = sheep = 0
        melon_young = 0
        for y in range(board):
            for x in range(board):
                t = farm["tiles"][y][x]
                if t == "LOCKED":
                    continue
                if t is None:
                    empty.append((x, y))
                elif isinstance(t, dict):
                    k = t.get("kind")
                    if k == "PLANT":
                        plants += 1
                        crop = t["crop"]
                        crop_count[crop] = crop_count.get(crop, 0) + 1
                        meta = CROP_META[crop]
                        age = day - t["planted_day"]
                        fert_now = t.get("fertilized_until_day", -1) >= day
                        if not t["watered_today"]:
                            # EVERY plant needs daily water (2 dry days = weed).
                            if crop == "MELON" and 6 <= age <= 11:
                                thirsty_melon_win.append((x, y))   # yield-critical days
                            elif meta["ongoing"]:
                                thirsty_ongoing.append((x, y))     # 2 dry days = death
                            else:
                                thirsty_other.append((x, y))
                        if crop == "MELON":
                            if age < 10:
                                melon_young += 1
                            if age >= 12:
                                harvest_melon.append((x, y))       # rot guard
                            elif age >= 10 and (t["yield_units"] >= 6
                                                or (age >= 11 and t["yield_units"] >= 5)
                                                or last_day or sellout):
                                harvest_melon.append((x, y))
                            if age == 6 and not fert_now and len(fert_melon) < 8:
                                fert_melon.append((x, y))          # window opener: +2/water
                        elif meta["ongoing"]:
                            if t["yield_units"] >= 2 or (t["yield_units"] > 0 and (last_day or sellout)):
                                harvest_ongoing.append((x, y))
                            lo = 6 if crop == "TOMATO" else 8
                            hi = 9 if crop == "TOMATO" else 14
                            if (not fert_now) and lo <= age <= hi and len(fert_ongoing) < 8:
                                fert_ongoing.append((x, y))
                            # v5 DEAD PIVOT: a standing crop whose market died
                            # is worth less than the tile replanted with a live
                            # crop. Price gates are generous (never dig on a
                            # hunch; only on measured collapse).
                            if DEAD_PIVOT and not last_day and t["yield_units"] <= 1:
                                p_now = prices.get(crop, BASE[crop])
                                dead_at = 25 if crop == "STRAWBERRY" else 20
                                if p_now < dead_at:
                                    dead_ongoing.append((x, y))
                        else:
                            if crop == "CARROT" and age >= 3 and t["yield_units"] > 0:
                                harvest_fast.append((x, y))
                            elif crop == "WHEAT" and age >= 4 and t["yield_units"] > 0:
                                harvest_fast.append((x, y))
                            if age == meta["first"] and not fert_now and len(fert_fast) < 12:
                                fert_fast.append((x, y))           # carrot/wheat window opener
                    elif k == "WEED":
                        weeds.append((x, y))
                    elif "animal" in t:
                        animals.append((x, y, t))
                        if t["animal"] == "GOOSE":
                            geese += 1
                        elif t["animal"] == "COW":
                            cows += 1
                        elif t["animal"] == "SHEEP":
                            sheep += 1
                    elif k == "COOP":
                        coops_empty.append((x, y))
                    elif k == "PASTURE":
                        pastures_empty.append((x, y))

        thirsty = thirsty_ongoing + thirsty_melon_win + thirsty_other
        unfed = [(x, y) for (x, y, t) in animals if not t["fed_today"]]
        uncared = [(x, y) for (x, y, t) in animals if t["fed_today"] and not t["cared_today"]]
        animal_ripe = [(x, y) for (x, y, t) in animals if t["yield_units"] > 0]
        fert_ready = [(x, y) for (x, y, t) in animals if t.get("fertilizer_available")]
        n_animals = len(animals)
        # v4 fert triage: melon openers (6-unit payoffs) outrank wheat/carrot
        # doublers (+3 units), which outrank ongoing speed-ups (+0 units at
        # cap - they only pull revenue forward).
        fert_targets = (fert_melon + fert_fast + fert_ongoing)[:12]

        # ------------------ SCOUT: melon headroom economics ------------------
        gap = max(0, inv.get("MELON", I0) - I0)
        opp_future = 0
        opp = farms[1 - player] if isinstance(player, int) and len(farms) > 1 - player else None
        if isinstance(opp, dict):
            ob = opp.get("tiles", [])
            for row in ob:
                for t in row:
                    if isinstance(t, dict) and t.get("kind") == "PLANT" \
                            and t.get("crop") == "MELON":
                        age = day - t.get("planted_day", day)
                        if age < 10:
                            opp_future += 5              # decent opponent: ~5-6 units each
                        elif OPP_RIPE and age < 12:
                            opp_future += 6              # RIPE: dumps within 48h, near-certain
        our_future = 5 * melon_young
        projected_gap = gap + our_future + opp_future
        melon_worth_planting = projected_gap < MELON_HEADROOM_CAP and day <= MELON_LAST_PLANT
        melon_market_dead = gap > 130                    # $1-20 forever; finish & pivot

        # ------------------ MARKET ORDERS ------------------
        shed_sum = sum(v for v in shed.values() if isinstance(v, (int, float)))
        orders = []

        # 1) MELON: the race. Every melon we own goes to market NOW.
        if shed.get("MELON", 0) > 0:
            orders.append(["SELL", "MELON", 999])

        # 2) RECOVERABLES: sell-all above floors (all zero); the only keeps
        #    are operational buffers (animal-feed wheat, window fertilizer).
        floor_mult = 1.0
        if shed_sum > 85:
            floor_mult = 0.5
        if money < 150:
            floor_mult = 0.3                  # emergency liquidity beats floors
        if sellout:
            floor_mult = 0.4 if day == SEASON_DAYS - 2 else 0.0
        keep_fert = KEEP_FERT if (fert_targets or crop_count["TOMATO"] or crop_count["STRAWBERRY"]) and not sellout else 0
        sell_batch = []
        for product in PRODUCTS:
            if product == "MELON":
                continue
            stock = int(shed.get(product, 0))
            if product == "WHEAT" and n_animals > 0 and not sellout:
                stock = max(0, stock - (n_animals + 3))       # keep feed buffer
            if product == "FERTILIZER":
                stock = max(0, stock - keep_fert)
            if product == "WOOL" and day >= WOOL_DUMP_DAY:
                stock = int(shed.get("WOOL", 0))              # dump wool late, no floor
            if stock <= 0:
                continue
            price = prices.get(product, 0)
            if price >= FLOOR[product] * BASE[product] * floor_mult or sellout \
                    or (product == "WOOL" and day >= WOOL_DUMP_DAY):
                cap = TRANCHE[product]
                if stock > 60 or shed_sum > 85:
                    cap *= 3
                if sellout and day >= SEASON_DAYS - 1:
                    cap = 999                                  # buzzer: everything, now
                sell_batch.append([price * stock, ["SELL", product, min(stock, cap)]])
        sell_batch.sort(key=lambda p: -p[0])
        sell_top = [o for _, o in sell_batch[:5 if ORDER_FULL else 3]]
        #    items beyond top-5 sell next TURN - same-day price is unchanged
        #    because the curve moves with cumulative supply, not clock time.
        #    (v5 bisect lesson: top-3 STARVED the shed - income vanished into
        #    EOD overflow. Sells are the engine; they never wait.)

        # 3) LABOR: re-hired every morning; scale with the workload. NO cash
        #    gate: the env itself refuses unaffordable hires, and fib wages
        #    ($1+$1+$2...) are so small that even a broke farm can re-staff.
        #    (v5 bisect lesson: capping hires at 1/turn broke the day-1
        #    bootstrap - hands ramp 3x slower and the whole season shrinks.
        #    Hires rank with sells in the order budget.)
        target_hands = _hand_target(plants, n_animals, day, money, last_day)
        more = target_hands - len(farm["hands"])
        if more > 0:
            orders += [["HIRE"]] * min(more, 3 if ORDER_FULL else 1)

        # 3b) SPEND PROJECTOR - every BUY this turn must leave projected
        #    money above the floor (v3.0 collapse lesson). v5 UNLOCK: the
        #    floor tracks the farm's ACTUAL wage bill - a 3-hand farm needs
        #    ~$150 of protection, not $400; the $400 veto is what kept the
        #    goose empire at 1-3 birds through the cash desert.
        wage_now = _wage_bill(len(farm["hands"]))
        floor_now = SPEND_FLOOR if not UNLOCK \
            else max(150, min(SPEND_FLOOR, wage_now + 150))
        projected = [money]

        def spend_ok(cost):
            if projected[0] - cost >= floor_now:
                projected[0] -= cost
                return True
            return False

        # 4) LAND - pace by utilization; land pays back in days, not weeks.
        land_cost = [1000, 2000, 4000][unlocked - 1] if unlocked < 4 else 0
        if unlocked < 4 and day <= LAND_LAST_DAY \
                and money >= land_cost + 1400 \
                and (plants >= 8 * unlocked or day >= 6) \
                and spend_ok(land_cost):
            orders.append(["BUY_LAND"])

        # 5) HERD - structure first (built below), then the bird/beast.
        #    Geese are the empire: 2 eggs/day + 1 free fert/day each (pillar 2
        #    fuel). Cows: milk is a SCARCITY market - the town drains it daily
        #    and almost nobody produces it (measured $208-386 vs base $160).
        #    A starved goose is a $300 stone: the herd only grows with the
        #    wheat engine that feeds it, plus the feed-buy line below (gate
        #    $60) as the bridge. Each animal costs ~4 task-actions/day
        #    (feed/care/harvest/collect) - the herd must fit the labor.
        n_geese = geese + int(shed.get("GOOSE", 0))
        n_cows = cows + int(shed.get("COW", 0))
        n_sheep = sheep + int(shed.get("SHEEP", 0))
        wheat_committed = crop_count["WHEAT"] + seeds.get("WHEAT", 0)
        goose_cap = min(GOOSE_BUDGET, 2 + 2 * wheat_committed) if not LIVESTOCK \
            else min(GOOSE_BUDGET, 4 + wheat_committed)
        goose_gate = 1000 if not UNLOCK else 500
        cow_gate = 1800 if not UNLOCK else 1200
        if n_geese < goose_cap and GOOSE_FIRST <= day <= GOOSE_LAST \
                and coops_empty and money >= goose_gate and spend_ok(300):
            orders.append(["BUY_ANIMAL", "GOOSE", 1])
        if n_cows < COW_BUDGET and COW_FIRST <= day <= COW_LAST \
                and pastures_empty and money >= cow_gate and spend_ok(400):
            orders.append(["BUY_ANIMAL", "COW", 1])
        if n_sheep < SHEEP_BUDGET and SHEEP_FIRST <= day <= SHEEP_LAST \
                and pastures_empty and money >= 1800 and spend_ok(500):
            orders.append(["BUY_ANIMAL", "SHEEP", 1])

        # 6) SEEDS - STAGED budgets so day-0 ambition never eats the bank;
        #    every buy also passes the SPEND_PROJECTOR above.
        def afford(cost):
            return money >= RESERVE + cost

        if not sellout:
            melon_budget = MELON_TARGET if day <= 10 else \
                (MELON_WAVE2 if melon_worth_planting else 0)
            melon_committed = crop_count["MELON"] + seeds.get("MELON", 0)
            if day <= MELON_LAST_PLANT and melon_worth_planting and melon_committed < melon_budget:
                n = min(melon_budget - melon_committed, 8)
                if n > 0 and afford(melon_budget * 80) and spend_ok(n * 80):
                    orders.append(["BUY_SEED", "MELON", n])
            straw_target = 6 if day <= 3 else (12 if day <= 8 else 16)
            straw_committed = crop_count["STRAWBERRY"] + seeds.get("STRAWBERRY", 0)
            if day <= LAST_SEED_DAY["STRAWBERRY"] and straw_committed < straw_target \
                    and afford(300):
                n = min(straw_target - straw_committed, 3)
                if n > 0 and spend_ok(n * 100):
                    orders.append(["BUY_SEED", "STRAWBERRY", n])
            toma_target = 0 if day < 3 else (4 if day <= 8 else 8)
            toma_committed = crop_count["TOMATO"] + seeds.get("TOMATO", 0)
            if day <= LAST_SEED_DAY["TOMATO"] and toma_committed < toma_target \
                    and afford(200):
                n = min(toma_target - toma_committed, 3)
                if n > 0 and spend_ok(n * 50):
                    orders.append(["BUY_SEED", "TOMATO", n])
            carrot_target = 6 if day <= 3 else 4
            carrot_committed = crop_count["CARROT"] + seeds.get("CARROT", 0)
            if day <= LAST_SEED_DAY["CARROT"] and carrot_committed < carrot_target \
                    and afford(100):
                n = min(carrot_target - carrot_committed, 5)
                if n > 0 and spend_ok(n * 20):
                    orders.append(["BUY_SEED", "CARROT", n])
            wheat_target = 6 if n_animals >= 4 else (2 if n_animals > 0 else 0)
            if UNLOCK and n_animals >= 4:
                wheat_target = WHEAT_TARGET_UNLOCK    # feed the unlocked flock
            if day <= LAST_SEED_DAY["WHEAT"] and wheat_committed < wheat_target \
                    and afford(150):
                n = min(wheat_target - wheat_committed, 3)
                if n > 0 and spend_ok(n * 10):
                    orders.append(["BUY_SEED", "WHEAT", n])

            # 6b) FLOW SEEDS (v5 pillar 1): keep the seed pantry stocked so
            #     every empty tile can be planted all season. The pantry
            #     thresholds ARE the flow regulator: buy a batch only when the
            #     pantry ran low, so seed inflow tracks actual planting speed.
            #     LAND-FIRST LAW: a flow batch is only bought when it cannot
            #     delay the next land purchase - land is 25 tiles at once and
            #     beats any flow batch (bisect lesson: un-gated flow flow
            #     starved the $2K/$4K land buys and LOST money).
            if FLOW_PLANT and day >= 4 and len(empty) > 0 and money >= 1500:
                land_cost_now = [1000, 2000, 4000][unlocked - 1] if unlocked < 4 else 0
                land_safe = unlocked >= 4 or money >= land_cost_now + 1400 + 400
                if land_safe:
                    if seeds.get("CARROT", 0) < 6 and spend_ok(100):
                        orders.append(["BUY_SEED", "CARROT", 5])
                    if seeds.get("WHEAT", 0) < 6 and spend_ok(50):
                        orders.append(["BUY_SEED", "WHEAT", 5])
                    if seeds.get("TOMATO", 0) < 4 and day <= 16 and spend_ok(150):
                        orders.append(["BUY_SEED", "TOMATO", 3])
                    if seeds.get("STRAWBERRY", 0) < 3 and day <= 14 \
                            and prices.get("STRAWBERRY", 0) >= 70 and spend_ok(200):
                        orders.append(["BUY_SEED", "STRAWBERRY", 2])

        # 7) FEED & FERTILIZER buys - feed keeps owned animals ALIVE (escape
        #    = $300 stone), so it outranks ambition; fert is free from geese,
        #    only bought on a goose-less farm with cash to burn.
        if n_animals > 0 and not sellout:
            shed_wheat = int(shed.get("WHEAT", 0))
            feed_need = n_animals if not LIVESTOCK else int(n_animals * 1.5) + 2
            feed_need = n_animals if not UNLOCK else int(n_animals * 1.25) + 2
            feed_gate_now = FEED_GATE if not UNLOCK else FEED_GATE_UNLOCK
            feed_cap_now = FEED_CAP if not UNLOCK else 16
            if shed_wheat < feed_need and prices.get("WHEAT", 99) <= feed_gate_now \
                    and money >= floor_now + 120:
                n_feed = min(feed_cap_now, feed_need - shed_wheat + 3)
                if spend_ok(n_feed * 30):
                    orders.append(["BUY_PRODUCT", "WHEAT", n_feed])
        if fert_targets and not sellout and int(shed.get("FERTILIZER", 0)) == 0 \
                and geese == 0 and prices.get("FERTILIZER", 999) <= 130 \
                and money >= 1500 and spend_ok(330):
            orders.append(["BUY_PRODUCT", "FERTILIZER", 3])

        # v5 ORDER BUDGET: the env processes at most 10 market orders per
        # player per turn and SILENTLY DROPS the rest. LAW (bisect-proven):
        # sells and hires NEVER wait; the cap is absorbed by the DEFERRABLE
        # classes (seeds, feed, animals, land), which simply retry next turn
        # - 24 tries a day, zero cost.
        melon_sells = [o for o in orders if o[0] == "SELL" and len(o) > 1 and o[1] == "MELON"]
        seed_agg = {}
        hire_orders, feed_orders, big_orders = [], [], []
        for o in orders:
            if o[0] == "BUY_SEED":
                seed_agg[o[1]] = seed_agg.get(o[1], 0) + o[2]
            elif o[0] == "HIRE":
                hire_orders.append(o)
            elif o[0] == "BUY_PRODUCT":
                feed_orders.append(o)
            elif o[0] in ("BUY_LAND", "BUY_ANIMAL"):
                big_orders.append(o)
            elif o[0] == "SELL" and len(o) > 1 and o[1] == "MELON":
                pass                                    # already in melon_sells
        SEED_PRIO = {"MELON": 0, "CARROT": 1, "STRAWBERRY": 2, "TOMATO": 3, "WHEAT": 4}
        seed_orders = [["BUY_SEED", c, n] for c, n in
                       sorted(seed_agg.items(), key=lambda kv: SEED_PRIO.get(kv[0], 9))]
        if ORDER_FULL:
            # Deferral-by-truncation: deferred classes self-retry next turn
            # (their pantry/afford conditions stay true), so a dropped seed
            # batch costs hours, not money. Worst turns (melon + 5 sells +
            # 3 hires + feed = 10) simply push seeds to a roomier turn.
            market_orders = (melon_sells + sell_top + hire_orders
                             + feed_orders[:1] + seed_orders + big_orders)[:10]
        else:
            market_orders = (melon_sells + sell_top + hire_orders[:1] + feed_orders
                             + seed_orders + big_orders)[:10]

        # ------------------ UNITS & TASK BOARD ------------------
        units = [("farmer", list(farm["farmer"]))] + \
                [(f"hand{i}", list(h)) for i, h in enumerate(farm["hands"])]

        tasks = []
        for xy in thirsty_melon_win:  tasks.append((2.1, xy, "WATER", None))
        for xy in thirsty_ongoing:    tasks.append((2.2, xy, "WATER", None))
        for xy in thirsty_other:      tasks.append((2.3, xy, "WATER", None))
        # v5 pillar 2: collect the free goose fertilizer EVERY day - $100 of
        #    product for one action. Priority sits BETWEEN the animal harvests
        #    and care: collecting must never delay a melon sale (the race) or
        #    a survival watering (bisect lesson: 2.5 starved both).
        #    QA G8 lesson: NO collects on the final day - pockets auto-drop to
        #    the shed AFTER the last market phase, so that fert can never be
        #    sold (1 unit unsold at the buzzer). The action is freed for
        #    real endgame work.
        if FERT_PIPELINE and not last_day:
            for xy in fert_ready:     tasks.append((3.45, xy, "COLLECT_FERTILIZER", None))
        for xy in harvest_melon:      tasks.append((3.0, xy, "HARVEST", None))
        for xy in harvest_fast:       tasks.append((3.1, xy, "HARVEST", None))
        for xy in harvest_ongoing:    tasks.append((3.2, xy, "HARVEST", None))
        for xy in animal_ripe:        tasks.append((ANIMAL_HARVEST_PRIO, xy, "HARVEST", None))
        for xy in uncared:            tasks.append((CARE_PRIO, xy, "CARE", None))
        if not FERT_PIPELINE:
            for xy in fert_ready:     tasks.append((3.6, xy, "COLLECT_FERTILIZER", None))
        # NOTE: FEED / FERTILIZE are carrier missions (need wheat/fert in
        # pockets) - see DISPATCH below.
        if melon_market_dead and (seeds.get("STRAWBERRY", 0) or seeds.get("CARROT", 0)):
            for y in range(board):
                for x in range(board):
                    t = farm["tiles"][y][x]
                    if isinstance(t, dict) and t.get("kind") == "PLANT" \
                            and t["crop"] == "MELON" and (day - t["planted_day"]) <= 8:
                        tasks.append((4.8, (x, y), "DIG", None))
        # v5 pillar 4: dig dead-market ongoing crops so the tile replants.
        if DEAD_PIVOT and dead_ongoing and (seeds.get("CARROT", 0) or seeds.get("WHEAT", 0)
                                            or seeds.get("TOMATO", 0) or seeds.get("STRAWBERRY", 0)):
            for xy in dead_ongoing[:8]:
                tasks.append((4.7, xy, "DIG", None))
        if hour <= 21 and not sellout:
            melon_cap_now = (MELON_TARGET if day <= 10 else MELON_WAVE2) \
                if melon_worth_planting else 0
            candidates_all = [c for c in ("MELON", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT")
                              if seeds.get(c, 0) > 0 and day <= LAST_SEED_DAY[c]]
            # v5 pillar 1 - FLOW PLANTING: v4 stopped planting once its small
            # cumulative targets were met, idling 60-94 tiles. Now every tile
            # gets the best-scoring crop the pantry can cover. Fixed targets
            # plant freely; beyond a target the plant counts against the
            # FLOW budget (keeps the mix balanced). Seeds are consumed at
            # dispatch (v4 semantics) - the pantry is the real throttle.
            fixed_now = {"MELON": melon_cap_now, "STRAWBERRY": straw_target,
                         "TOMATO": toma_target, "CARROT": carrot_target,
                         "WHEAT": wheat_target}
            flow_budget = FLOW_DAILY_CAP if FLOW_PLANT else 0
            flow_planned = [0]
            for xy in empty:
                if not candidates_all:
                    break
                melon_left = melon_cap_now - crop_count["MELON"]
                if "MELON" in candidates_all and melon_left <= 0:
                    cands = [c for c in candidates_all if c != "MELON"]
                else:
                    cands = candidates_all
                if not cands:
                    break
                fert_ok = bool(fert_targets) or geese > 0 or int(shed.get("FERTILIZER", 0)) > 0

                def crop_score(c):
                    m = CROP_META[c]
                    p = prices.get(c, BASE[c])
                    if c == "MELON":
                        end_gap = min(projected_gap + 30, 200)
                        p_eff = max(1.0, (price_above("MELON", gap) + price_above("MELON", end_gap)) / 2)
                        return (m["max"] * p_eff - m["seed"]) / 11.0 - (5.0 if day > 10 else 0.0)
                    if c == "STRAWBERRY":
                        rate = 0.75 if fert_ok else 0.4     # 4 events x 2 units / ~10 d
                        days = max(1.0, SEASON_DAYS - day - m["first"])
                        return rate * p * 0.85 - m["seed"] / days
                    if c == "TOMATO":
                        rate = 1.5 if fert_ok else 0.75     # 4 events x 2 units / ~5 d
                        days = max(1.0, SEASON_DAYS - day - m["first"])
                        return rate * p * 0.85 - m["seed"] / days
                    if c == "CARROT":
                        return (4.0 * p - m["seed"]) / 4.0  # fert-doubled
                    return (6.0 * p - m["seed"]) / 5.0 + (1.5 if n_animals > 0 else 0.0)
                best = max(cands, key=crop_score)
                committed = crop_count.get(best, 0) + seeds.get(best, 0)
                if FLOW_PLANT and committed >= fixed_now.get(best, 0):
                    # beyond the fixed target this is a FLOW plant
                    if flow_planned[0] >= flow_budget:
                        continue
                    flow_planned[0] += 1
                tasks.append((5.0, xy, "PLANT", best))
        for xy in weeds:              tasks.append((8.0, xy, "DIG", None))
        # housing for the next animal (free, one action, near the shed)
        if not sellout and empty:
            if n_geese < GOOSE_BUDGET and not coops_empty and GOOSE_FIRST <= day <= GOOSE_LAST:
                spot = min(empty, key=lambda p: abs(p[0] - 4) + abs(p[1] - 4))
                tasks.append((4.5, spot, "BUILD_COOP", None))
            elif (n_cows + n_sheep) < (COW_BUDGET + SHEEP_BUDGET) and not pastures_empty \
                    and COW_FIRST <= day <= SHEEP_LAST:
                spot = min(empty, key=lambda p: abs(p[0] - 4) + abs(p[1] - 4))
                tasks.append((4.5, spot, "BUILD_PASTURE", None))

        all_pos = [pos for _, pos in units]
        tasks.sort(key=lambda t: (t[0], min(abs(t[1][0] - px) + abs(t[1][1] - py)
                                            for px, py in all_pos)))

        # ------------------ DISPATCH (zones + greedy nearest) ------------------
        invs = private.get("inventories", [])
        carry_wheat = {u: (invs[i].get("WHEAT", 0) if i < len(invs) else 0) for i, (u, _) in enumerate(units)}
        carry_fert = {u: (invs[i].get("FERTILIZER", 0) if i < len(invs) else 0) for i, (u, _) in enumerate(units)}
        carry_animal = {u: ((invs[i].get("GOOSE", 0) or 0) + (invs[i].get("COW", 0) or 0)
                            + (invs[i].get("SHEEP", 0) or 0)
                            if i < len(invs) else 0) for i, (u, _) in enumerate(units)}
        carry_any = {u: (sum(v for v in invs[i].values() if isinstance(v, (int, float)))
                         if i < len(invs) else 0) for i, (u, _) in enumerate(units)}
        carry_mel = {u: (invs[i].get("MELON", 0) if i < len(invs) else 0) for i, (u, _) in enumerate(units)}

        zone_names = ["NW", "NE", "SW", "SE"]
        zones = {u: (zone_names[i % 4] if i <= 4 else None) for i, (u, _) in enumerate(units)}

        free = {u: pos for u, pos in units}
        plan = {u: ["PASS"] for u, _ in units}

        def pick_unit(tx, ty):
            best, best_key = None, None
            for u, (fx, fy) in free.items():
                z = zones[u]
                in_zone = (z is None) or (_quadrant(tx, ty) == z)
                key = (0 if in_zone else 1, abs(fx - tx) + abs(fy - ty))
                if best_key is None or key < best_key:
                    best, best_key = u, key
            return best

        # --- missions that must claim workers before the task board ---
        #  a) fetch wheat for feeding
        if unfed and sum(carry_wheat.values()) == 0 and int(shed.get("WHEAT", 0)) > 0:
            for u, pos in units:
                if _at_shed(pos):
                    plan[u] = ["PICKUP", "WHEAT", min(10, int(shed.get("WHEAT", 0)))]
                    del free[u]
                    break
            else:
                u = pick_unit(4, 4)
                if u:
                    fx, fy = free[u]
                    step = _step_toward(fx, fy, 4, 4)
                    if step:
                        plan[u] = [step]
                        del free[u]
        #  a2) FEED mission: wheat CARRIERS serve unfed animals.
        for xy in unfed:
            carriers = [u for u in free if carry_wheat.get(u, 0) > 0]
            if not carriers:
                break
            u = min(carriers, key=lambda u: abs(free[u][0] - xy[0]) + abs(free[u][1] - xy[1]))
            fx, fy = free[u]
            if (fx, fy) == (xy[0], xy[1]):
                plan[u] = ["FEED"]
                carry_wheat[u] -= 1
            else:
                step = _step_toward(fx, fy, xy[0], xy[1])
                plan[u] = [step] if step else ["PASS"]
            del free[u]
        #  b) fetch fertilizer for the yield windows
        if fert_targets and sum(carry_fert.values()) == 0 and int(shed.get("FERTILIZER", 0)) > 0:
            for u, pos in units:
                if _at_shed(pos) and u in free:
                    plan[u] = ["PICKUP", "FERTILIZER", min(6, int(shed.get("FERTILIZER", 0)))]
                    del free[u]
                    break
            else:
                u = pick_unit(4, 4)
                if u:
                    fx, fy = free[u]
                    step = _step_toward(fx, fy, 4, 4)
                    if step:
                        plan[u] = [step]
                        del free[u]
        #  b2) FERTILIZE mission: fert CARRIERS serve the yield windows.
        for xy in fert_targets:
            carriers = [u for u in free if carry_fert.get(u, 0) > 0]
            if not carriers:
                break
            u = min(carriers, key=lambda u: abs(free[u][0] - xy[0]) + abs(free[u][1] - xy[1]))
            fx, fy = free[u]
            if (fx, fy) == (xy[0], xy[1]):
                plan[u] = ["FERTILIZE"]
                carry_fert[u] -= 1
            else:
                step = _step_toward(fx, fy, xy[0], xy[1])
                plan[u] = [step] if step else ["PASS"]
            del free[u]
        #  c) deliver a bought animal (from the shed OR from a pocket that is
        #     mid-delivery) to its structure. v5 QA fix: the job used to exist
        #     only while the animal sat in the shed, so the moment a carrier
        #     picked it up the job vanished and the hot-drop rule marched the
        #     carrier back to the shed - a PICKUP/DROP oscillation that left
        #     birds unplaced at the buzzer. The job now tracks pocket-held
        #     animals until PLACE completes.
        pocket_animal = {}
        if POCKET_JOB:
            for i, (u, _) in enumerate(units):
                if i < len(invs):
                    for k in ("GOOSE", "COW", "SHEEP"):
                        if (invs[i].get(k, 0) or 0) > 0 and u not in pocket_animal:
                            pocket_animal[u] = k
        animal_job = None
        if (int(shed.get("GOOSE", 0)) > 0 or any(p == "GOOSE" for p in pocket_animal.values())) and coops_empty:
            animal_job = ("GOOSE", coops_empty[0])
        elif (int(shed.get("COW", 0)) > 0 or any(p == "COW" for p in pocket_animal.values())) and pastures_empty:
            animal_job = ("COW", pastures_empty[0])
        elif (int(shed.get("SHEEP", 0)) > 0 or any(p == "SHEEP" for p in pocket_animal.values())) and pastures_empty:
            animal_job = ("SHEEP", pastures_empty[0])
        if animal_job:
            item, (tx, ty) = animal_job
            holder = next((u for u, p in pocket_animal.items() if p == item), None) if POCKET_JOB \
                else next((u for u, _ in units if carry_animal.get(u, 0) > 0), None)
            if holder and holder in free:
                fx, fy = free[holder]
                if (fx, fy) == (tx, ty):
                    plan[holder] = ["PLACE", item]
                else:
                    step = _step_toward(fx, fy, tx, ty)
                    plan[holder] = [step] if step else ["PASS"]
                del free[holder]
            elif not holder:
                u = pick_unit(4, 4)
                if u:
                    fx, fy = free[u]
                    if _at_shed((fx, fy)):
                        plan[u] = ["PICKUP", item, 1]
                    else:
                        step = _step_toward(fx, fy, 4, 4)
                        plan[u] = [step] if step else ["PASS"]
                    del free[u]
        #  d) hot-drop: cargo must reach the shed to be sellable. Guaranteed
        #     on the final day; before that MELONS always, animals, fertilizer
        #     batches (>=4, pillar 2), and heavy pockets; only if the shed has
        #     room.
        if shed_sum < 96:
            hot = last_day or sellout
            for u, pos in units:
                if u not in free or carry_any.get(u, 0) <= 0:
                    continue
                wants_drop = hot or carry_animal.get(u, 0) > 0 \
                    or carry_mel.get(u, 0) > 0 or carry_any[u] >= 6 \
                    or (FERT_PIPELINE and carry_fert.get(u, 0) >= 4)
                if not wants_drop:
                    continue
                if _at_shed(pos):
                    plan[u] = ["DROP"]
                else:
                    tx, ty = min(SHED_TILES, key=lambda s: abs(pos[0] - s[0]) + abs(pos[1] - s[1]))
                    step = _step_toward(pos[0], pos[1], tx, ty)
                    plan[u] = [step] if step else ["PASS"]
                del free[u]

        # --- main task loop: most urgent task claims the best free unit ---
        for prio, (tx, ty), op, arg in tasks:
            if not free:
                break
            u = pick_unit(tx, ty)
            fx, fy = free[u]
            if (fx, fy) == (tx, ty):
                if op == "PLANT":
                    if arg and seeds.get(arg, 0) > 0:
                        plan[u] = ["PLANT", arg]
                        seeds[arg] -= 1
                        crop_count[arg] = crop_count.get(arg, 0) + 1
                    else:
                        continue
                elif op == "FEED":
                    if carry_wheat.get(u, 0) > 0:
                        plan[u] = ["FEED"]
                        carry_wheat[u] -= 1
                    else:
                        continue
                elif op == "FERTILIZE":
                    if carry_fert.get(u, 0) > 0:
                        plan[u] = ["FERTILIZE"]
                        carry_fert[u] -= 1
                    else:
                        continue
                else:
                    plan[u] = [op]
            else:
                step = _step_toward(fx, fy, tx, ty)
                if not step:
                    continue
                plan[u] = [step]
            del free[u]

        hands = [plan.get(f"hand{i}", ["PASS"]) for i in range(len(farm["hands"]))]
        return {"farmer": plan.get("farmer", ["PASS"]), "hands": hands, "market": market_orders}

    except Exception as e:
        # Grandmaster rule #1: a crashed agent scores zero for the episode.
        # Fail over to a blind full-liquidation (SELL aborts harmlessly on
        # empty stock) so even a bad turn still banks everything sellable.
        if _ERR["n"] < 3:
            _ERR["n"] += 1
            print(f"[farm_os] fallback #{_ERR['n']}: {type(e).__name__}: {e}")
        out = {"farmer": ["PASS"], "hands": [], "market": []}
        try:
            shed = (obs.get("private", {}) or {}).get("shed", {}) or {}
            for item, stock in shed.items():
                if isinstance(stock, (int, float)) and stock > 0 and len(out["market"]) < 10:
                    out["market"].append(["SELL", item, 999])
        except Exception:
            pass
        return out


# Kaggle entry point
def agent(obs):
    return farm_os(obs)


In [ ]:
# Season 3, harvest day: load the Empress straight from the submission file
# (single source of truth: what you test here is EXACTLY what gets submitted).
from submission import farm_os

def show(a, b, label):
    env = make("kaggriculture", debug=True)
    env.run([a, b])
    r0, r1 = env.steps[-1][0].reward, env.steps[-1][1].reward
    verdict = "WIN" if r0 > r1 else ("LOSS" if r0 < r1 else "TIE")
    print(f"{label:14s} ${r0:>8,.0f}  vs  ${r1:>8,.0f}   -> {verdict}")

show(farm_os, "random",             "vs random")
show(farm_os, "starter",            "vs starter")
show(farm_os, melon_maxxer,         "vs A1")
show(farm_os, diversified_farmer,   "vs A2")
show(farm_os, farm_os,              "vs itself")

## 🚀 Chapter 4 — Your Name on the Ladder

The cell above wrote your whole farm into **`submission.py`** — one self-contained file, zero imports, nothing to install. Three ways to enter the town contest:

1. **From this notebook** — click **"Submit to Competition"** (right panel). Because this notebook *contains* `submission.py`, it qualifies.
2. **Upload a file** — hit *Submit Agents* on the competition page and upload `submission.py` (or a `.tar.gz` holding it).
3. **Kaggle CLI** — scriptable submissions when you iterate daily (you get **5 per day**; only your latest 2 are tracked; your best farm shows on the board).

Every submission first plays a **validation episode against itself** — crash there and the submission is marked `Error` with downloadable logs.

**About that number on the ladder:** Kaggle scores farms with a Bradley-Terry rating — the chess-club Elo of farming. Beat a strong farm, gain a lot; beat a weak one, gain a sliver; lose the reverse. Because who you're matched against is partly luck, your score **wobbles** — a few points up one morning, a few down the next (our v1 opened at ~358, v3 hit ~439, v4 reached ~470 — each by the same wobble-and-trend path). Don't chase single numbers: submit, let it play 20–30 games, and judge the *trend*. Every engine in this notebook moved the trend in **local tournaments on fresh, unseen random seeds against six opponent classes before touching the leaderboard** — that habit is your protection against overfitting to the public board.

**Public and private boards — how selection actually works:** during the competition your agent plays public ladder games and the board shows your best score. At the deadline, Kaggle re-scores on held-out private games, and **your best-scoring submission is what counts** — so there is nothing extra to configure, but there IS one rule: make every submission count, keep the strongest one alive (don't let your final daily submissions be reckless experiments), and prefer changes validated the boring way — more local wins, bigger margins, across many seeds — over any change that only looked good once. An agent tuned to yesterday's public opponents is a farmer who memorized last year's weather; an agent validated across unseen seeds and opponent classes wins both boards.

> **The pre-submission checklist (each line is a war story)**
>
> - ✅ **Never crash.** Every agent here wraps its whole body in `try/except` returning a `PASS` — a crashed bot scores 0 *for the entire episode*.
> - ✅ **Never go to $0.** No coins = no morning hires = no watering = weeds in 2 days = death spiral (the PURSE engine exists for this).
> - ✅ **Carry-what-you-use.** `FEED`/`FERTILIZE` consume from *pockets*; a worker without cargo silently no-ops while your geese starve.
> - ✅ **Water plantings the same day** (a fresh seed starts with 1 unwatered day already counted).
> - ✅ **Never over-issue `PLANT`** beyond seed count — the env voids *all* of them that turn.
> - ✅ **Mind the barn cap (100)** — dusk overflow is *discarded*, not queued.
> - ✅ **Sell before the buzzer** — day-30 sweep, no exceptions.
> - ✅ **Hands vanish nightly** — re-hire every morning; it's practically free.
> - ✅ **Sell everything NOW (the telescoping law)** — waiting provably earns $0; keep only feed wheat and window fertilizer.
> - ✅ **Labor is zero-sum (Law 5)** — before adding any chore, ask what it displaces; the v5 audit rejected four profitable-looking ideas for exactly this reason.

### Where your story goes next

| Direction | The idea | Expected payoff |
|---|---|---|
| 🌡️ Price prediction | track `market.inventory` deltas across turns to *anticipate* floors before selling | sell closer to local peaks |
| 🐑 The sheep question | wool pays $200/unit — but it's a quadratic mini-melon; pace the tranches | diversification depth |
| ⚔️ Rival memory | Farm OS reads the rival's farm each turn; next: track *their* sell patterns and race them for whichever market they ignore | ladder-edge decisions |
| 🧪 Self-play tuning | mirror matches + grid-search `MELON_TARGET`, `MELON_HEADROOM_CAP`, floors, herd budgets | +10–20% typical |
| 🎛️ Host overrides | the host can override `marketParams` per episode — read prices, don't hardcode *behavior* | robustness |

**The mirror-match moral.** Two naive melon-maxxers strangle the same finite market together. Two Farm OS copies read the depth and split it — both bank ~$35K. On the real ladder the field is your mirror: **read what everyone else is flooding, then be the farmer who prices it correctly.**

Good luck out there, cousin. Uncle Robert is watching. 🌾